# Underwater YOLO11 — ***C2PSA_ParallelCAFM*** Training / Validation (Colab Template v4)

This notebook:

- Installs **Ultralytics** from a **specific GitHub branch** in the fork
- Extracts `UOD_Dataset.tar` from **Google Drive** to Colab’s local storage
- Saves runs + checkpoints to **Google Drive**, so you can **resume** after idle timeouts

**Workflow:** To change `MODEL_CONFIG_YAML`, `ABLATION_NAME`, and `BRANCH_NAME` in the configuration cell for each ablation.

## 0) Configuration

Central settings for experiment naming, storage paths, training parameters, augmentations, and required hardware. Review these values before running each ablation.

In [1]:
from google.colab import runtime
import albumentations as A
import torch
from pathlib import Path

# The ablation experiment identifier
ABLATION_NAME = "c2psa-parallel-cafm"

# Set the git branch name to correspond to the ablation name
BRANCH_NAME = "ablation/c2psa-parallel-cafm"
BASE_DIR = Path("/content/drive/MyDrive/UnderWaterYOLO11/")

# Set ultralytics project name to correspond to the ablation name
PROJECT_NAME = BASE_DIR / ABLATION_NAME

RUN_NAME = 'train'
VAL_RUN_NAME = 'val'
DATASET_ROOT = Path("/content/UOD_Dataset")
DATA_YAML = DATASET_ROOT / "data_split.yaml"

# COCO annotation file for faster-coco-eval
COCO_ANN_FILE = DATASET_ROOT / "annotations" / "instances_val2017.json"

# Training parameters
IMG_SIZE = 640
EPOCHS   = 300
BATCH    = 64
PATIENCE = 0
MODEL_CONFIG_YAML = "yolo11m-c2psa-parallel-cafm.yaml"  # baseline / ablation architecture YAML file
MODEL_PRETRAIN = "yolo11m.pt"       # pre-training checkpoint weights for finetuning

# Albumentations Transforms
T = [
    A.MedianBlur(p=0.01),
    A.CLAHE(p=0.01)
]

# Ultralytics built-in augmentations
AUGMENT_KWARGS = {
    # Photometric augmentations
    "hsv_h": 0.0,
    "hsv_s": 0.4,
    "hsv_v": 0.2,

    # Geometric augmentations
    "scale": 0.0,
    "translate": 0.0,

    # Mosaic
    "mosaic": 0.2,
    "close_mosaic": 50
}

REQUIRED_GPU = 'NVIDIA RTX PRO 6000 Blackwell Server Edition' # 8.71 units per hour
# REQUIRED_GPU = 'NVIDIA A100-SXM4-40GB' # 5.44 units per hour
# REQUIRED_GPU = 'NVIDIA A100-SXM4-80GB' # 7.52 units per hour
# REQUIRED_GPU = None
gpu_name = torch.cuda.get_device_name(0)
if REQUIRED_GPU and gpu_name != REQUIRED_GPU:
    print('❌ This notebook must execute on the google Colab with runtime type: ', REQUIRED_GPU)
    runtime.unassign()
else:
    print('✅ Running on', gpu_name)

✅ Running on NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1) Mount Google Drive and set a Drive-backed experiment directory
All training artifacts (including `last.pt`) are written to Google Drive.


In [2]:
# 📂 Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Create base directory if it does not exist
BASE_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 2) Install and import Ultralytics from fork + branch

In [3]:
%pip install "git+https://github.com/marcofarrugia/ultralytics.git@{BRANCH_NAME}"

import ultralytics
from ultralytics import YOLO
from ultralytics.utils import YAML

ultralytics.checks()

Ultralytics 8.3.244 [Underwater YOLO11 Ablations (c2psa-parallel-cafm) by Marco Farrugia] 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 46.8/235.7 GB disk)


## 3) Extract the dataset from Google Drive

The dataset is extracted from `UOD_Dataset.tar` inside `BASE_DIR`, to Colab's local `/content` storage for faster access during training. The dataset directory and its `data_split.yaml` configuration file are identified for use by the remaining cells.

In [4]:
import tarfile
from tqdm.auto import tqdm

DATASET_ARCHIVE = BASE_DIR / "UOD_Dataset.tar"

if not DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found:\n{DATASET_ARCHIVE}"
    )

# Extract only when the dataset is not already available.
if not DATA_YAML.is_file():
    print(f"Transferring dataset from: {DATASET_ARCHIVE}")

    with tarfile.open(DATASET_ARCHIVE, "r:*") as archive:
        members = archive.getmembers()
        archive.extractall(
            path=DATASET_ROOT.parent,
            members=tqdm(
                members,
                total=len(members),
                desc="Extracting dataset",
                unit="file",
                dynamic_ncols=True,
            ),
            filter="data",
        )

if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Dataset configuration file not found after extraction: {DATA_YAML}"
    )

print("Dataset extracted to:", DATASET_ROOT)
print("Dataset configuration:", DATA_YAML)

Transferring dataset from: /content/drive/MyDrive/UnderWaterYOLO11/UOD_Dataset.tar


Extracting dataset:   0%|          | 0/33256 [00:00<?, ?file/s]

Dataset extracted to: /content/UOD_Dataset
Dataset configuration: /content/UOD_Dataset/data_split.yaml


## 4) Train with automatic resume
- If a previous run exists in the same (Google Drive) folder, `resume=True` continues training

In [5]:
def should_resume() -> tuple[bool, Path | None]:
    run_dir = PROJECT_NAME / RUN_NAME
    last_ckpt = run_dir / "weights" / "last.pt"
    args_yaml = run_dir / "args.yaml"
    if not args_yaml.exists():
        return False, None
    args = YAML.load(args_yaml) or {}
    return not args.get("completed", False), last_ckpt

def mark_completed(trainer) -> None:
    args_yaml = Path(trainer.save_dir) / "args.yaml"
    args = YAML.load(args_yaml) or {}
    args["completed"] = True
    YAML.save(args_yaml, args)

resume, last_ckpt = should_resume()
# resume = False           # use for experimenting only: Always start from scratch

model = None

if resume:
    print(f"Resuming from checkpoint:\n{last_ckpt}")
    model = YOLO(last_ckpt)
elif last_ckpt:
    print("Training skipped as it has already been completed in a previous run")
else:
    print(f"Starting fresh training run on {MODEL_CONFIG_YAML} using pretrained weights from {MODEL_PRETRAIN}")
    model = YOLO(MODEL_CONFIG_YAML).load(MODEL_PRETRAIN)

if model:
    model.add_callback("on_train_end", mark_completed)

    model.train(
        data=DATA_YAML,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH,
        project=PROJECT_NAME,
        name=RUN_NAME,
        patience=PATIENCE,
        resume=resume,
        pretrained=True,
        augmentations = T,
        **AUGMENT_KWARGS,       # augmentation settings
        exist_ok = True,
        # fraction = 0.1        # use for experimenting only: (10% of the dataset will be used)
    )


Starting fresh training run on yolo11m-c2psa-parallel-cafm.yaml using pretrained weights from yolo11m.pt
Transferred 649/658 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.244 [Underwater YOLO11 Ablations (c2psa-parallel-cafm) by Marco Farrugia] 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, augmentations=[MedianBlur(p=0.01, blur_limit=(3, 7)), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))], auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=50, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/UOD_Dataset/data_split.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=

## 5) Generate COCO ground-truth annotations from the validation split YOLO labels

Only the labels referenced by the validation manifest from `DATA_YAML` are supplied to PyLabel, while the original image and label filenames remain unchanged. PyLabel assigns the numeric image IDs used by the COCO ground-truth file. Prediction IDs are aligned with these IDs by filename and written to a separate `predictions_coco_ids.json` file.

In [6]:
%pip install "pylabel==0.1.55" -q --progress-bar on

from shutil import copy2
from tempfile import TemporaryDirectory
from pylabel import importer

COCO_ANN_FILE.parent.mkdir(parents=True, exist_ok=True)

if COCO_ANN_FILE.exists():
    print(f"COCO annotations already exist → skipping export:\n{COCO_ANN_FILE}")
else:
    data_config = YAML.load(DATA_YAML)
    val_manifest = DATASET_ROOT / data_config["val"]
    print(f"Generating COCO annotations from YOLO labels: {val_manifest}")

    dataset_dir = val_manifest.parent
    labels_dir = dataset_dir / "labels"

    # PyLabel receives only labels belonging to the validation manifest.
    with TemporaryDirectory(
        prefix="coco_val_labels_",
        dir=dataset_dir,
    ) as temp_dir:
        temp_labels_dir = Path(temp_dir)

        for image_ref in val_manifest.read_text(encoding="utf-8").splitlines():
            label_name = f"{Path(image_ref).stem}.txt"
            copy2(
                labels_dir / label_name,
                temp_labels_dir / label_name,
            )

        importer.ImportYoloV5(
            path=str(temp_labels_dir),
            path_to_images="../images/",
            cat_names=data_config["names"],
            img_ext="jpg,jpeg,png",
        ).export.ExportToCoco(
            cat_id_index=1,
            output_path=str(COCO_ANN_FILE),
        )

    print(f"\nCOCO annotations written to: {COCO_ANN_FILE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.9 MB/s eta 0:00:00
Generating COCO annotations from YOLO labels: /content/UOD_Dataset/dataset/val.txt


Importing YOLO files...: 100%|██████████| 1663/1663 [00:09<00:00, 174.01it/s]
/usr/local/lib/python3.12/dist-packages/pylabel/exporter.py:767: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r"^\s*$", np.nan, regex=True)
Exporting to COCO file...: 100%|██████████| 8328/8328 [00:01<00:00, 6121.85it/s]


COCO annotations written to: /content/UOD_Dataset/annotations/instances_val2017.json


## 6) Validate the best model

In [5]:
# Load best checkpoint
best_ckpt = PROJECT_NAME / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_ckpt)

# Run validation
val_results = best_model.val(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH * 2, # A larger batch size potentially improves GPU utilisation and reduces per-batch overhead
    split="val",
    project=PROJECT_NAME,
    name=VAL_RUN_NAME,
    save_json=True,
    exist_ok=True,  # Replace any previous validation results
    verbose=True
)

Ultralytics 8.3.244 [Underwater YOLO11 Ablations (c2psa-parallel-cafm) by Marco Farrugia] 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO11m-c2psa-parallel-cafm summary (fused): 130 layers, 20,386,292 parameters, 0 gradients, 68.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7274.4±2232.2 MB/s, size: 239.1 KB)
val: Scanning /content/UOD_Dataset/dataset/labels... 1663 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1663/1663 6.3Kit/s 0.3s
val: New cache created: /content/UOD_Dataset/dataset/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.3it/s 5.6s
                   all       1663       8328      0.595      0.436      0.496      0.346
               bivalve        228        770      0.414      0.278      0.258      0.169
                bottle        157        294      0.458       0.18      0.242       0.13
            cephalopod   

## 7) Evaluate Validation Results

### 7.1 Align Prediction IDs with COCO Ground Truth

Reference: [Ultralytics issue #21557 — incorrect `image_id` conversion](https://github.com/ultralytics/ultralytics/issues/21557)

In [8]:
import json

save_dir = Path(val_results.save_dir)
pred_json = save_dir / "predictions.json"
aligned_pred_json = save_dir / "predictions_coco_ids.json"

# Map filenames to PyLabel-generated COCO image IDs.
with open(COCO_ANN_FILE, "r", encoding="utf-8") as f:
    coco_annotations = json.load(f)

coco_id_by_filename = {
    Path(image["file_name"]).name: image["id"]
    for image in coco_annotations["images"]
}

# Align every Ultralytics prediction with the PyLabel ground-truth image ID.
with open(pred_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

for prediction in predictions:
    filename = Path(prediction["file_name"]).name
    prediction["image_id"] = coco_id_by_filename[filename]

with open(aligned_pred_json, "w", encoding="utf-8") as f:
    json.dump(predictions, f)

### 7.2 Compute Validation Metrics

In [9]:
%pip install "faster-coco-eval==1.7.2" -q --progress-bar on

from faster_coco_eval import COCO, COCOeval_faster
from ultralytics.utils.torch_utils import get_flops, get_num_params

# -------------------------------
# Ultralytics metrics
# -------------------------------

# Inference timing
latency_inference = val_results.speed["inference"]

# Total inference pipeline latency per image, excluding validation-only loss computation
latency_end_to_end = sum(
    val_results.speed[key]
    for key in ("preprocess", "inference", "postprocess")
)

# Raw model speed
fps_inference = 1000.0 / latency_inference

# Real application throughput (including data handling)
fps_end_to_end = 1000.0 / latency_end_to_end

num_params = get_num_params(best_model.model)
flops = get_flops(best_model.model, imgsz=IMG_SIZE)

# -------------------------------
# COCO metrics
# -------------------------------
coco_gt = COCO(COCO_ANN_FILE)
coco_dt = coco_gt.loadRes(aligned_pred_json)

coco_eval = COCOeval_faster(
    coco_gt,
    coco_dt,
    iouType="bbox",
    print_function=lambda *args, **kwargs: None,
)
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

stats = coco_eval.stats_as_dict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 6.7 MB/s eta 0:00:00


### 7.3 Display Validation Metrics

In [10]:
metrics_order = [
    ("AP_all", "mAP@0.50:0.95"),
    ("AP_50", "mAP@0.50"),
    ("AP_75", "mAP@0.75"),
    ("AP_small", "mAP (Small)"),
    ("AP_medium", "mAP (Medium)"),
    ("AP_large", "mAP (Large)"),
    ("AR_third", "AR@0.50:0.95 (maxDets=100)"),
    ("AR_small", "AR (Small)"),
    ("AR_medium", "AR (Medium)"),
    ("AR_large", "AR (Large)"),
    # ("AR_all", "AR@0.50:0.95 (maxDets=1)"),
    # ("AR_second", "AR@0.50:0.95 (maxDets=10)"),
    # ("AR_50", "AR@0.50"),
    # ("AR_75", "AR@0.75")
]

# Pretty print the metrics
print("\n=============== VALIDATION ACCURACY RESULTS ===============")
for key, label in metrics_order:
    value = stats.get(key, float("nan"))
    print(f"{label:<26} | {value:.4f}")
print("\n=============== VALIDATION SPEED RESULTS ===============")
print(f"{'Inference Latency (forward pass only)':<37} | {latency_inference:.2f} ms / image")  # forward pass only, no pre/postprocessing
print(f"{'Inference Latency (end-to-end)':<37} | {latency_end_to_end:.2f} ms / image")        # preprocessing + inference + postprocessing
print(f"{'Inference Speed (forward pass only)':<37} | {fps_inference:.2f} FPS")
print(f"{'Inference Speed (end-to-end)':<37} | {fps_end_to_end:.2f} FPS")
print(f"{'Parameters':<37} | {num_params:,}")
print(f"{'GFLOPs':<37} | {flops:.1f}")
print("==================================================\n")



=============== VALIDATION ACCURACY RESULTS ===============
mAP@0.50:0.95              | 0.3040
mAP@0.50                   | 0.4510
mAP@0.75                   | 0.3101
mAP (Small)                | 0.0972
mAP (Medium)               | 0.2174
mAP (Large)                | 0.3420
AR@0.50:0.95 (maxDets=100) | 0.3892
AR (Small)                 | 0.1477
AR (Medium)                | 0.2930
AR (Large)                 | 0.4267

=============== VALIDATION SPEED RESULTS ===============
Inference Latency (forward pass only) | 1.28 ms / image
Inference Latency (end-to-end)        | 1.79 ms / image
Inference Speed (forward pass only)   | 781.35 FPS
Inference Speed (end-to-end)          | 558.55 FPS
Parameters                            | 20,386,292
GFLOPs                                | 68.0



### 7.4 Inspect Learned CAFM Gates

In [11]:
from math import isfinite

def inspect_gammas(weights_path: str | Path) -> dict[str, float]:
    weights_path = Path(weights_path)
    detector = YOLO(str(weights_path)).model

    expected_names = {
        f"model.10.m.{index}.gamma"
        for index in range(len(detector.model[10].m))
    }
    gammas = {}

    for name, parameter in detector.named_parameters():
        if name in expected_names:
            if parameter.numel() != 1:
                raise RuntimeError(
                    f"{name} should be scalar, but has shape {tuple(parameter.shape)}"
                )

            value = parameter.detach().float().cpu().item()
            if not isfinite(value):
                raise RuntimeError(f"{name} is not finite: {value}")

            gammas[name] = value

    if set(gammas) != expected_names:
        raise RuntimeError(
            f"Expected {sorted(expected_names)}, found {sorted(gammas)}"
        )

    print(f"\n{weights_path.name}")
    for name, value in sorted(gammas.items()):
        print(f"{name}: {value:+.8f}")

    return gammas


best_gammas = inspect_gammas(best_ckpt)


best.pt
model.10.m.0.gamma: -0.00168896


## 8) Terminate the current runtime to preserve GPU Time

In [6]:
print('✅ Mission accomplished')
runtime.unassign()

✅ Mission accomplished
